# Flight Price Prediction - Raw Data Inspection

Goal of this notebook: confirm `flight_sample.csv` looks correct before doing any feature engineering.

We check:
- schema and dtypes
- row count
- missing values
- date ranges (searchDate, flightDate)
- route coverage (should be exactly our 20 chosen routes)

No feature engineering or modeling happens here, just verification.

In [1]:
import polars as pl

pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_width_chars(200)

df = pl.read_csv("../flight_sample.csv")
df.shape

(1500000, 27)

## Schema and dtypes

Check that Polars inferred sensible types on load (dates as strings not auto-parsed, numeric columns as numeric, booleans as booleans).

In [2]:
for name, dtype in df.schema.items():
    print(f"{name:40s} {dtype}")

legId                                    String
searchDate                               String
flightDate                               String
startingAirport                          String
destinationAirport                       String
fareBasisCode                            String
travelDuration                           String
elapsedDays                              Int64
isBasicEconomy                           Boolean
isRefundable                             Boolean
isNonStop                                Boolean
baseFare                                 Float64
totalFare                                Float64
seatsRemaining                           Int64
totalTravelDistance                      Int64
segmentsDepartureTimeEpochSeconds        String
segmentsDepartureTimeRaw                 String
segmentsArrivalTimeEpochSeconds          String
segmentsArrivalTimeRaw                   String
segmentsArrivalAirportCode               String
segmentsDepartureAirportCode          

## Missing values

Count nulls per column. Some segment columns (e.g. equipment description) may legitimately have gaps; totalTravelDistance had at least one null in the raw preview too.

In [3]:
null_counts = df.null_count().transpose(include_header=True, header_name="column", column_names=["null_count"])
null_counts = null_counts.with_columns((pl.col("null_count") / df.height * 100).round(2).alias("null_pct"))
null_counts.sort("null_count", descending=True)

column,null_count,null_pct
str,u32,f64
"""totalTravelDistance""",155912,10.39
"""segmentsEquipmentDescription""",46136,3.08
"""legId""",0,0.0
"""searchDate""",0,0.0
"""flightDate""",0,0.0
"""startingAirport""",0,0.0
"""destinationAirport""",0,0.0
"""fareBasisCode""",0,0.0
"""travelDuration""",0,0.0


## Date ranges

searchDate and flightDate are what let us honestly compute days-until-departure later. Check both ranges make sense and searchDate is always <= flightDate.

In [4]:
df_dates = df.with_columns([
    pl.col("searchDate").str.to_date(),
    pl.col("flightDate").str.to_date(),
])

print("searchDate range:", df_dates["searchDate"].min(), "to", df_dates["searchDate"].max())
print("flightDate range:", df_dates["flightDate"].min(), "to", df_dates["flightDate"].max())

rows_where_search_after_flight = df_dates.filter(pl.col("searchDate") > pl.col("flightDate")).height
print("Rows where searchDate > flightDate (should be 0):", rows_where_search_after_flight)

searchDate range: 2022-04-16 to 2022-10-05
flightDate range: 2022-04-17 to 2022-11-19
Rows where searchDate > flightDate (should be 0): 0


## Route coverage

Confirm the sample contains exactly the 20 directed routes we filtered for (10 city-pairs, both directions), and see how evenly they're represented.

In [5]:
route_counts = (
    df.group_by(["startingAirport", "destinationAirport"])
    .len()
    .sort("len", descending=True)
)
print("Distinct routes in sample:", route_counts.height)
route_counts

Distinct routes in sample: 20


startingAirport,destinationAirport,len
str,str,u32
"""ATL""","""LAX""",86988
"""LGA""","""LAX""",82952
"""LAX""","""BOS""",82789
"""LAX""","""ATL""",81805
"""LAX""","""LGA""",81015
"""BOS""","""LAX""",78774
"""LAX""","""JFK""",76774
"""LAX""","""ORD""",75756
"""DFW""","""LAX""",75111
